# CDDFuse Variant Training Template

Inputs: `harvard-medical-train` (738) + `harvard-medical-fusion` (72) + `cddfuse-pretrained`.

## Cell 1 â€” Config

In [ ]:
VARIANT     = 'Combined-Gated-Saliency'
REPO_URL    = 'https://github.com/kienvbhp872004/Image-Fusion.git'
REPO_BRANCH = 'main'
REPO_COMMIT = ''
EPOCHS      = 25
BATCH       = 4
SEED        = 42

import os, subprocess
os.environ['VARIANT'] = VARIANT

r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                   capture_output=True, text=True)
GPU_NAME = r.stdout.strip().splitlines()[0] if r.stdout else 'unknown'
print(f'[gpu] {GPU_NAME}')
# P100 = sm_60 → cần downgrade torch 2.5.1+cu121 (Cell 2). Sau đó GPU mode hoạt động.
DEVICE = 'auto'  # train.py sẽ pick CUDA nếu được
os.environ['EPOCHS'] = str(EPOCHS)
os.environ['BATCH']  = str(BATCH)
os.environ['DEVICE'] = DEVICE

## Cell 2 â€” Clone repo, install deps

In [ ]:
!git clone --branch $REPO_BRANCH $REPO_URL /kaggle/working/Image-Fusion
%cd /kaggle/working/Image-Fusion
if REPO_COMMIT:
    !git checkout $REPO_COMMIT
!pip install -q -r requirements.txt
!pip install -q kornia tqdm

# Downgrade torch để support P100 sm_60 (torch 2.6+ drop Pascal binary).
# torch 2.5.1+cu121 là bản cuối hỗ trợ sm_60. Subprocess !python sau sẽ pick bản mới.
!pip uninstall -y torch torchvision torchaudio 2>&1 | tail -3
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!python -c "import torch; print('[torch]', torch.__version__, 'cuda:', torch.cuda.is_available(), 'cap:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'N/A')"

## Cell 3 â€” Stage datasets (auto-locate path)

In [ ]:
import shutil, zipfile, pathlib, glob

def find_dataset_root(slug):
    for c in [f'/kaggle/input/{slug}', f'/kaggle/input/datasets/kienvbhp1234/{slug}']:
        if pathlib.Path(c).exists():
            return pathlib.Path(c)
    matches = glob.glob(f'/kaggle/input/**/{slug}', recursive=True)
    if matches: return pathlib.Path(matches[0])
    raise FileNotFoundError(slug)

def stage_modal(src_root, modal, dst_root):
    dst = dst_root / modal
    dst.mkdir(parents=True, exist_ok=True)
    folder = src_root / modal
    zip_p  = src_root / f'{modal}.zip'
    if folder.exists() and folder.is_dir():
        shutil.copytree(folder, dst, dirs_exist_ok=True); return 'folder'
    if zip_p.exists():
        with zipfile.ZipFile(zip_p) as zf: zf.extractall(dst)
        return 'zip'
    raise FileNotFoundError(f'{folder} or {zip_p}')

TRAIN_SRC = find_dataset_root('harvard-medical-train')
TEST_SRC  = find_dataset_root('harvard-medical-fusion')
CKPT_SRC  = find_dataset_root('cddfuse-pretrained')
print(f'[paths] train={TRAIN_SRC}\n[paths] test ={TEST_SRC}\n[paths] ckpt ={CKPT_SRC}')

TRAIN_OUT = pathlib.Path('/kaggle/working/Image-Fusion/data/train')
for modal in ['CT-MRI','PET-MRI','SPECT-MRI']:
    m = stage_modal(TRAIN_SRC, modal, TRAIN_OUT)
    print(f'[train] {modal} ({m}): {len(list((TRAIN_OUT/modal).rglob("*.png")))} files')

TEST_OUT = pathlib.Path('/kaggle/working/Image-Fusion/data/reference')
for modal in ['CT-MRI','PET-MRI','SPECT-MRI']:
    m = stage_modal(TEST_SRC, modal, TEST_OUT)
    print(f'[test ] {modal} ({m}): {len(list((TEST_OUT/modal).rglob("*.png")))} files')

CKPT_DIR = pathlib.Path('/kaggle/working/Image-Fusion/models/MMIF-CDDFuse/models')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(CKPT_SRC / 'CDDFuse_MIF.pth', CKPT_DIR)
print(f'[ckpt ] -> {CKPT_DIR}/CDDFuse_MIF.pth')

## Cell 4 â€” Train

In [ ]:
%cd /kaggle/working/Image-Fusion/models/MMIF-CDDFuse
!python -m variants.train \
    --variant     $VARIANT \
    --pretrained  models/CDDFuse_MIF.pth \
    --train_data  /kaggle/working/Image-Fusion/data/train \
    --output      /kaggle/working/ \
    --epochs      $EPOCHS \
    --batch       $BATCH \
    --seed        $SEED \
    --device      $DEVICE

## Cell 5 â€” Inference + per-image metrics

In [ ]:
OUT_DIR = f'/kaggle/working/CDDFuse-{VARIANT}'
CKPT    = f'/kaggle/working/CDDFuse-{VARIANT}.pth'
TEST    = '/kaggle/working/Image-Fusion/data/reference'

for modal in ['CT', 'PET', 'SPECT']:
    !python evaluate_cddfuse.py \
        --variant       $VARIANT \
        --modal         $modal \
        --ckpt          $CKPT \
        --harvard_root  $TEST \
        --out_dir       $OUT_DIR \
        --device        $DEVICE \
        --save_perimage

## Cell 6 â€” Package output

In [ ]:
import tarfile, json, hashlib, datetime, os
if not os.path.exists(CKPT):
    raise RuntimeError(f'No ckpt at {CKPT} - Cell 4 failed.')
h = hashlib.sha256()
with open(CKPT, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''): h.update(chunk)

stamp = {
    'variant_name':   f'CDDFuse-{VARIANT}',
    'based_on':       'CDDFuse',
    'datetime':       datetime.datetime.utcnow().isoformat() + 'Z',
    'train_mode':     'light_retrain',
    'device':         DEVICE,
    'gpu_detected':   GPU_NAME,
    'epochs_phase2':  EPOCHS,
    'batch_size':     BATCH,
    'seed':           SEED,
    'checkpoint':     {'sha256': h.hexdigest()},
}
with open(f'{OUT_DIR}/_ablation_stamp.json', 'w') as f:
    json.dump(stamp, f, indent=2)

tar_path = f'/kaggle/working/CDDFuse-{VARIANT}_results.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(OUT_DIR, arcname=f'CDDFuse-{VARIANT}')
    tar.add(CKPT, arcname=f'CDDFuse-{VARIANT}.pth')
    hist = f'/kaggle/working/CDDFuse-{VARIANT}_train_history.json'
    if os.path.exists(hist):
        tar.add(hist, arcname=f'CDDFuse-{VARIANT}_train_history.json')
print('[done]', tar_path)